# 💧 Water Quality Data Analysis

---

## 📌 Project Overview
This notebook analyzes a Water Quality Dataset (`BKB_WaterQualityData_2020084.csv`) to understand historical trends in various environmental metrics such as Salinity, Dissolved Oxygen, pH, and Temperature. 

### Objectives
- Perform initial Exploratory Data Analysis (EDA) and assess missing values.
- Clean the dataset by handling nulls and ensuring valid numerical ranges.
- Visualize time-series trends for key water quality indicators.
- Generate a correlation matrix to identify relationships between metrics.
- Create an interactive geospatial map of sampling points using Folium.
- Detect abnormal water quality conditions based on established environmental thresholds.

---
## 1. Setup & Library Imports
We import essential Python libraries for data manipulation, visualization, and mapping.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import folium
import os

# Optional: List files in the Kaggle input directory
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

---
## 2. Data Loading & Initial Exploration
Load the dataset and examine its structure, data types, and the extent of missing values across all columns.

In [ ]:
# Load the dataset
file_path = "/kaggle/input/water-quality-data/BKB_WaterQualityData_2020084.csv"  # Update if running locally
try:
    data = pd.read_csv(file_path)
except FileNotFoundError:
    print(f"File not found at {file_path}. Please update the path.")

# Display the first few rows
print("Dataset Overview:")
display(data.head())

# Display dataset information
print("\nDataset Info:")
data.info()

# Check for missing values
print("\nMissing Values Count:")
print(data.isnull().sum())

---
## 3. Data Preprocessing & Cleaning
We format the dates, remove records missing critical measurements, and filter out invalid negative values to ensure a high-quality dataset for our analysis.

In [ ]:
# Convert Read_Date to datetime format
data['Read_Date'] = pd.to_datetime(data['Read_Date'])

# Remove rows with missing critical values
data = data.dropna(subset=[
    'Salinity (ppt)', 'Dissolved Oxygen (mg/L)', 'pH (standard units)', 
    'Secchi Depth (m)', 'Water Depth (m)', 'Water Temp (?C)', 'Air Temp-Celsius'
])

# Ensure numeric columns are non-negative where applicable
numeric_cols = [
    'Salinity (ppt)', 'Dissolved Oxygen (mg/L)', 'Secchi Depth (m)', 
    'Water Depth (m)', 'Water Temp (?C)', 'Air Temp-Celsius'
]

for col in numeric_cols:
    data = data[data[col] >= 0]

# Verify changes
print("Cleaned Data Overview:")
data.info()

---
## 4. Summary Statistics
Analyzing the distribution, mean, minimum, and maximum for our cleaned numerical columns.

In [ ]:
# Summary statistics for numeric columns
print("\nSummary Statistics:")
display(data.describe())

---
## 5. Visualizing Trends & Correlations
Time-series analysis and a correlation heatmap to discover how variables like Salinity, Water Temperature, and Dissolved Oxygen behave over time and in relation to one another.

In [ ]:
# Set index for time series plotting without overwriting original dataframe structure permanently
data_ts = data.set_index('Read_Date')

# Time series plot for key metrics
data_ts[['Salinity (ppt)', 'Dissolved Oxygen (mg/L)', 'Water Temp (?C)']].plot(figsize=(12, 6))
plt.title("Water Quality Trends Over Time")
plt.ylabel("Measurements")
plt.xlabel("Date")
plt.grid()
plt.legend()
plt.show()

In [ ]:
# Heatmap for correlation between numerical features
plt.figure(figsize=(10, 6))
sns.heatmap(data[numeric_cols].corr(), annot=True, cmap='coolwarm', fmt=".2f")
plt.title("Correlation Matrix of Water Quality Metrics")
plt.show()

---
## 6. Geospatial Mapping
Using `folium` to plot data points on an interactive map. *(Note: Using placeholder coordinates here since exact lat/lon coordinates are absent from this specific extract).*.

In [ ]:
# Example coordinates for location center (Placeholder for average latitude and longitude)
map_center = [40.7128, -74.0060]  
water_quality_map = folium.Map(location=map_center, zoom_start=6)

# Add data points to the map (Limiting to 100 markers to keep the map lightweight)
for _, row in data.head(100).iterrows():
    folium.CircleMarker(
        location=map_center,  # Replace map_center with [row['Lat'], row['Lon']] if you have coordinate columns
        radius=5,
        popup=f"Salinity: {row['Salinity (ppt)']} ppt\nDissolved Oxygen: {row['Dissolved Oxygen (mg/L)']} mg/L",
        color='blue',
        fill=True,
        fill_color='blue'
    ).add_to(water_quality_map)

# Save map as HTML
water_quality_map.save("water_quality_map.html")

# Display the map directly in the notebook
water_quality_map

---
## 7. Anomaly Detection
Filtering the dataset to identify days/locations where water quality fell outside acceptable environmental thresholds.

In [ ]:
# Thresholds for abnormal measurements
low_dissolved_oxygen = 3  # mg/L
high_salinity = 35        # ppt

# Filter data for abnormal conditions
abnormal_conditions = data[
    (data['Dissolved Oxygen (mg/L)'] < low_dissolved_oxygen) | 
    (data['Salinity (ppt)'] > high_salinity)
]

print(f"\nFound {len(abnormal_conditions)} abnormal condition records.")

if len(abnormal_conditions) > 0:
    print("\nAbnormal Water Quality Conditions:")
    # Displaying the records (Using 'Site_Id' as the identifier)
    display(abnormal_conditions[['Site_Id', 'Unit_Id', 'Salinity (ppt)', 'Dissolved Oxygen (mg/L)', 'Read_Date']])
else:
    print("No abnormal conditions detected in the cleaned subset.")